# AIT-ADS baseline comparison

Compares eleven system-task baseline families evaluated on one AIT-ADS scenario's train/test split -- the AIT-ADS counterpart to `notebooks/baselines/cscas_baseline_comparison.ipynb` (see `baselines/_ait_ads_data.py`, `baselines/ait_ads_rf.py`, `baselines/ait_ads_logreg.py`, `baselines/ait_ads_xgboost.py`, `baselines/ait_ads_bert.py`, `baselines/ait_ads_securebert.py`, `baselines/ait_ads_zeroshot.py`, `baselines/ait_ads_anomaly.py`, `baselines/ait_ads_anomaly_iforest.py`, `baselines/ait_ads_mining.py`, `baselines/ait_ads_mining_anomaly.py`, `baselines/ait_ads_mining_anomaly_iforest.py`):

1. **Base schema (RF)** -- 5-feature `base` schema, `RandomForestClassifier`
2. **Base schema (LogReg)** -- same, `StandardScaler` + `LogisticRegression`
3. **Base schema (XGBoost)** -- same, `XGBClassifier`
4. **BERT** -- alert_group tokens (`sig:`/`host:`/`short:`) serialized to text, fine-tuned DistilBERT
5. **SecureBERT 2.0** -- same tokens, fine-tuned SecureBERT 2.0 (ModernBERT architecture, domain-adapted)
6. **Zero-shot** -- same tokens serialized to a prompt, no fine-tuning
7. **Anomaly (OneClassSVM)** -- same 5-feature base schema, `OneClassSVM` fit on benign-only train rows -- see the exclusion paragraph below for why this one is included as its own method rather than folded into the other six.
8. **Anomaly (IsolationForest)** -- same 5-feature base schema, benign-only fit, second anomaly-detector model family alongside method 7 -- isolates model choice within the anomaly-detector family the same way LogReg/XGBoost do within the classifier family. Tree-based, so unlike OneClassSVM it needs no feature scaling.
9. **Base schema + Mining (RF / LogReg / XGBoost)** -- the same 5-feature base schema extended with symbolic features mined by the attribute-mining pipeline (contrast-set + decision-tree rules) on the identical train split, fit with all three tabular classifiers (the mining siblings of methods 1-3, and the AIT-ADS counterpart of CSCAS's `cscas_mining.py`). One attribute-mining pass per `(scenario, grouping)` feeds all three; results are saved as `ait_ads_mining_*` (RF), `ait_ads_mining_logreg_*` and `ait_ads_mining_xgboost_*`.
10. **Anomaly + Mining (OneClassSVM)** -- method 9's mined symbolic features, on top of the reduced base schema, `OneClassSVM` fit on benign-only train rows -- the mining sibling of method 7, and the most complete "system" scenario (mining + task, one-class) evaluated here.
11. **Anomaly + Mining (IsolationForest)** -- method 9's mined symbolic features on the reduced base schema, `IsolationForest` fit on benign-only train rows -- the IsolationForest sibling of method 10 (and of method 8), 5-seeded (`random_state=0..4`).

**Two training-pool conditions, not three**: random undersampling and class-weighted (natural-ratio). No "guided" condition here -- AIT-ADS has no SCAS-equivalent outlier signal to guide sampling with (CSCAS-only, see `baselines/_sampling.py`'s own docstring). Zero-shot and all four anomaly methods have no training-pool step, so none of the conditions apply to them. Zero-shot and the two **OneClassSVM** anomaly methods are single deterministic runs (OneClassSVM has no `random_state`; its per-seed sd is exactly 0); the two **IsolationForest** anomaly methods are seed-averaged over 5 seeds like the classifiers. Every anomaly method reports `auc` alongside precision/recall/f1, plus `workload_at_recall` -- precision / FP / analyst-workload-reduction at the threshold hitting each target recall -- since the default `nu`/`contamination` = 0.05 cut over-flags relative to the true attack prevalence.

All methods are **seed-averaged (`N_SEEDS = 5`)** except zero-shot and the two OneClassSVM anomaly methods (single deterministic runs -- zero-shot at temperature=0, OneClassSVM has no `random_state` to average over). The two IsolationForest anomaly methods run the same 5-seed protocol (`random_state=0..4`), same convention as the CSCAS scripts.

**All ten methods share the exact same split** -- every script calls `_ait_ads_data.load_ait_ads_baseline_split(scenario, grouping_method)` (the two mining scripts use `load_ait_ads_baseline_split_with_groups`, the same split plus the AlertGroup objects mining needs), so results are directly comparable across model families for a given `(SCENARIO, GROUPING_METHOD)`, not just similarly configured pipelines. This replaced an earlier approach that pulled RF/LogReg/XGBoost from a separate, fixed_window-only pipeline (`run_model_comparison_attribute.py`) -- that could only ever guarantee *similar* splits, not identical ones, and couldn't be extended to the other 4 grouping methods without duplicating this same split logic anyway.

Results are **per (grouping method, scenario)** -- unlike CSCAS (pregrouped), AIT-ADS alerts need a grouping step first, and which of the 5 grouping methods (`fixed_window`, `time_delta`, `cscas_grouping`, `alertbert`, `deepcase`) is used is itself an axis, not a fixed choice (see `baselines/_ait_ads_grouping.py`). Set `SCENARIO` and `GROUPING_METHOD` in the Settings cell below.

**`alertbert`/`deepcase` are only valid for `fox`, `harrison`, `russellmitchell`, `santos`** -- both the pretrained AlertBERT checkpoint and the DeepCASE ContextBuilder were trained on `shaw`/`wardbeck`/`wheeler`/`wilson`, so grouping those 4 scenarios with either method for a baseline result would be self-training leakage. All ten AIT-ADS baseline scripts skip that combination automatically.

**`harrison` and `santos` produce no result under any grouping method, and `russellmitchell` only under `deepcase`, for the seven *classifier* methods (1-6, 9 above)** -- a different, unrelated reason: the train/test split is a fixed chronological cut (last `test_frac` of the timeline is test, no search for a "valid" boundary), and for these 3 scenarios every attack-labelled alert_group falls within that cut, leaving train 100% benign regardless of grouping method. This is an accepted exclusion, not a bug -- see `baselines/_ait_ads_data.py`'s module docstring for why the split isn't adjusted to recover them (doing so would give them a different effective train/test ratio than the other 5 scenarios). **The four anomaly methods (7, 8, 10, 11) are exempt**: all are fit on benign rows only, so a 100%-benign train split is exactly what they need, not a blocker -- `ait_ads_anomaly.py`/`ait_ads_anomaly_iforest.py`/`ait_ads_mining_anomaly.py`/`ait_ads_mining_anomaly_iforest.py` use a test-side-only single-class guard instead of the train-side one the other seven scripts use. For method 10 specifically, mining itself still runs on the (100%-benign) train split -- with no attack rows to contrast against, `run_alert_group_attribute_mining_job` mines 0 predicates rather than raising, so these 3 scenarios fall back to methods 10/11 reporting the same numbers as methods 7/8 there, still a valid result, just without added symbolic signal (still subject to the alertbert/deepcase leakage skip above where applicable).

**Run the scripts first** to generate the `results/*.json` files this notebook reads:
```
cd src/thesis/baselines
python ait_ads_rf.py
python ait_ads_logreg.py
python ait_ads_xgboost.py
python ait_ads_bert.py
python ait_ads_securebert.py
OLLAMA_MODEL=llama3.1:8b python ait_ads_zeroshot.py
python ait_ads_anomaly.py
python ait_ads_anomaly_iforest.py
python ait_ads_mining.py          # fits RF + LogReg + XGBoost on the mined matrix
python ait_ads_mining_anomaly.py
python ait_ads_mining_anomaly_iforest.py
```

In [1]:
from __future__ import annotations

import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from thesis.baselines._results import RESULTS_DIR, is_anomaly, is_zero_shot, load_baseline_results
from thesis.paths import CACHE_DIR

## Settings

Edit `SCENARIO` / `GROUPING_METHOD` (and `ZEROSHOT_MODEL_SLUGS` if the zero-shot sweep used a different model set) and re-run -- nothing past this cell needs to change.

In [2]:
SCENARIO = "fox"  # fox, harrison, russellmitchell, santos, shaw, wardbeck, wheeler, wilson
GROUPING_METHOD = "fixed_window"  # fixed_window, time_delta, cscas_grouping, alertbert, deepcase
# alertbert/deepcase are only valid for fox/harrison/russellmitchell/santos -- see the intro cell above.

# Zero-shot sweep -- OLLAMA_MODEL with ":"/"/" replaced by "-", the MODELS list in
# shell-scripts/baselines/run_ait_ads_zeroshot.sh. All four appear in every table;
# any whose results/*.json isn't present just render as blank rows.
ZEROSHOT_MODEL_SLUGS = ["llama3.1-8b", "llama3.1-70b", "qwen2.5-7b", "qwen2.5-72b"]
_ZS = [f"ait_ads_zeroshot_{GROUPING_METHOD}_{SCENARIO}_{s}" for s in ZEROSHOT_MODEL_SLUGS]

RESULT_NAMES = [
    f"ait_ads_rf_{GROUPING_METHOD}_{SCENARIO}",
    f"ait_ads_logreg_{GROUPING_METHOD}_{SCENARIO}",
    f"ait_ads_xgboost_{GROUPING_METHOD}_{SCENARIO}",
    f"ait_ads_bert_{GROUPING_METHOD}_{SCENARIO}",
    f"ait_ads_securebert_{GROUPING_METHOD}_{SCENARIO}",
    *_ZS,
    f"ait_ads_anomaly_{GROUPING_METHOD}_{SCENARIO}_ocsvm",
    f"ait_ads_anomaly_{GROUPING_METHOD}_{SCENARIO}_iforest",
    f"ait_ads_mining_{GROUPING_METHOD}_{SCENARIO}",
    f"ait_ads_mining_logreg_{GROUPING_METHOD}_{SCENARIO}",
    f"ait_ads_mining_xgboost_{GROUPING_METHOD}_{SCENARIO}",
    f"ait_ads_mining_anomaly_{GROUPING_METHOD}_{SCENARIO}_ocsvm",
    f"ait_ads_mining_anomaly_{GROUPING_METHOD}_{SCENARIO}_iforest",
]

LABELS = {
    f"ait_ads_rf_{GROUPING_METHOD}_{SCENARIO}": f"Base schema (RF, {GROUPING_METHOD})",
    f"ait_ads_logreg_{GROUPING_METHOD}_{SCENARIO}": f"Base schema (LogReg, {GROUPING_METHOD})",
    f"ait_ads_xgboost_{GROUPING_METHOD}_{SCENARIO}": f"Base schema (XGBoost, {GROUPING_METHOD})",
    f"ait_ads_bert_{GROUPING_METHOD}_{SCENARIO}": f"BERT (tokens as text, {GROUPING_METHOD})",
    f"ait_ads_securebert_{GROUPING_METHOD}_{SCENARIO}": f"SecureBERT 2.0 (tokens as text, {GROUPING_METHOD})",
    **{k: f"Zero-shot ({s}, {GROUPING_METHOD})" for k, s in zip(_ZS, ZEROSHOT_MODEL_SLUGS)},
    f"ait_ads_anomaly_{GROUPING_METHOD}_{SCENARIO}_ocsvm": f"Anomaly (OneClassSVM, benign-only, {GROUPING_METHOD})",
    f"ait_ads_anomaly_{GROUPING_METHOD}_{SCENARIO}_iforest": f"Anomaly (IsolationForest, benign-only, {GROUPING_METHOD})",
    f"ait_ads_mining_{GROUPING_METHOD}_{SCENARIO}": f"Base schema + Mining (RF, {GROUPING_METHOD})",
    f"ait_ads_mining_logreg_{GROUPING_METHOD}_{SCENARIO}": f"Base schema + Mining (LogReg, {GROUPING_METHOD})",
    f"ait_ads_mining_xgboost_{GROUPING_METHOD}_{SCENARIO}": f"Base schema + Mining (XGBoost, {GROUPING_METHOD})",
    f"ait_ads_mining_anomaly_{GROUPING_METHOD}_{SCENARIO}_ocsvm": f"Anomaly + Mining (OneClassSVM, benign-only, {GROUPING_METHOD})",
    f"ait_ads_mining_anomaly_{GROUPING_METHOD}_{SCENARIO}_iforest": f"Anomaly + Mining (IsolationForest, benign-only, {GROUPING_METHOD})",
}

COLORS = {
    f"ait_ads_rf_{GROUPING_METHOD}_{SCENARIO}": "#CCBB44",
    f"ait_ads_logreg_{GROUPING_METHOD}_{SCENARIO}": "#228833",
    f"ait_ads_xgboost_{GROUPING_METHOD}_{SCENARIO}": "#66CCEE",
    f"ait_ads_bert_{GROUPING_METHOD}_{SCENARIO}": "#EE6677",
    f"ait_ads_securebert_{GROUPING_METHOD}_{SCENARIO}": "#AA3377",
    **{k: c for k, c in zip(_ZS, ["#BBBBBB", "#999999", "#DDDDDD", "#777777"])},
    f"ait_ads_anomaly_{GROUPING_METHOD}_{SCENARIO}_ocsvm": "#44BB99",
    f"ait_ads_anomaly_{GROUPING_METHOD}_{SCENARIO}_iforest": "#117733",
    f"ait_ads_mining_{GROUPING_METHOD}_{SCENARIO}": "#EE8866",
    f"ait_ads_mining_logreg_{GROUPING_METHOD}_{SCENARIO}": "#DDAA77",
    f"ait_ads_mining_xgboost_{GROUPING_METHOD}_{SCENARIO}": "#CC9955",
    f"ait_ads_mining_anomaly_{GROUPING_METHOD}_{SCENARIO}_ocsvm": "#DDCC77",
    f"ait_ads_mining_anomaly_{GROUPING_METHOD}_{SCENARIO}_iforest": "#999933",
}

# Scoped per (scenario, grouping method) so different combinations' figures/summaries don't overwrite each other.
FIGURES_DIR = RESULTS_DIR / "figures" / SCENARIO / GROUPING_METHOD
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
results = {}
for name in RESULT_NAMES:
    try:
        results[name] = load_baseline_results(name)
    except FileNotFoundError as e:
        print(f"[skip] {e}")

print(f"Loaded results for: {list(results.keys())}")

[skip] /Users/annavisman/stack/TUDelft/thesis/msc-thesis/src/thesis/baselines/results/ait_ads_zeroshot_fixed_window_fox_llama3.1-8b.json not found -- run `python ait_ads_zeroshot_fixed_window_fox_llama3.1-8b.py` from src/thesis/baselines/ first.
[skip] /Users/annavisman/stack/TUDelft/thesis/msc-thesis/src/thesis/baselines/results/ait_ads_zeroshot_fixed_window_fox_llama3.1-70b.json not found -- run `python ait_ads_zeroshot_fixed_window_fox_llama3.1-70b.py` from src/thesis/baselines/ first.
[skip] /Users/annavisman/stack/TUDelft/thesis/msc-thesis/src/thesis/baselines/results/ait_ads_zeroshot_fixed_window_fox_qwen2.5-7b.json not found -- run `python ait_ads_zeroshot_fixed_window_fox_qwen2.5-7b.py` from src/thesis/baselines/ first.
[skip] /Users/annavisman/stack/TUDelft/thesis/msc-thesis/src/thesis/baselines/results/ait_ads_zeroshot_fixed_window_fox_qwen2.5-72b.json not found -- run `python ait_ads_zeroshot_fixed_window_fox_qwen2.5-72b.py` from src/thesis/baselines/ first.
Loaded results f

In [4]:
def plot_baseline(condition_key: str, title: str) -> None:
    """Grouped bar chart across every loaded method that actually reports
    `condition_key` -- excludes zero-shot and anomaly always (both flat
    shape, no conditions -- see plot_zeroshot()/plot_anomaly() below).
    Saves to FIGURES_DIR/{condition_key}.png before displaying."""
    metrics = ["precision", "recall", "f1"]
    methods = [
        n for n in RESULT_NAMES
        if n in results and not is_zero_shot(results[n]) and not is_anomaly(results[n])
        and condition_key in results[n]
    ]
    if not methods:
        print(f"No results loaded with a '{condition_key}' condition -- run the baseline scripts first.")
        return

    x = np.arange(len(metrics))
    width = 0.85 / len(methods)

    fig, ax = plt.subplots(figsize=(10, 5))
    for i, name in enumerate(methods):
        values = [results[name][condition_key][m] for m in metrics]
        offset = (i - (len(methods) - 1) / 2) * width
        bars = ax.bar(
            x + offset, values, width,
            label=LABELS.get(name, name), color=COLORS.get(name),
        )
        ax.bar_label(bars, fmt="%.2f", fontsize=7, padding=2, rotation=90)

    ax.set_xticks(x)
    ax.set_xticklabels([m.capitalize() for m in metrics])
    ax.set_ylim(0, 1.2)
    ax.set_ylabel("Score")
    ax.set_title(f"{title} ({SCENARIO}, {GROUPING_METHOD})")
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=3, fontsize=8)
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()

    save_path = FIGURES_DIR / f"{condition_key}.png"
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    print(f"Figure written to {save_path}")

    plt.show()

## Summary table

Tidy view of all loaded results for this scenario -- handy to copy straight into a slide.

In [5]:
CONDITION_LABELS = {
    "random": "Random undersampling",
    "class_weighted": "Class-weighted (natural-ratio)",
}

rows = []
for name, data in results.items():
    if is_zero_shot(data) or is_anomaly(data):
        rows.append({
            "method": LABELS.get(name, name),
            "condition": "(no training)",
            "auc": data.get("auc"),
            "precision": data["precision"],
            "recall": data["recall"],
            "f1": data["f1"],
        })
    else:
        for condition_key, condition_label in CONDITION_LABELS.items():
            if condition_key not in data:
                continue
            rows.append({
                "method": LABELS.get(name, name),
                "condition": condition_label,
                "auc": None,
                "precision": data[condition_key]["precision"],
                "recall": data[condition_key]["recall"],
                "f1": data[condition_key]["f1"],
            })

if not rows:
    print("No results loaded -- run the baseline scripts first.")
    summary_df = pd.DataFrame(columns=["method", "condition", "auc", "precision", "recall", "f1"])
else:
    summary_df = pd.DataFrame(rows).sort_values(["condition", "method"]).reset_index(drop=True)
    summary_csv_path = RESULTS_DIR / f"ait_ads_{SCENARIO}_{GROUPING_METHOD}_baseline_summary.csv"
    summary_df.to_csv(summary_csv_path, index=False)
    print(f"Summary written to {summary_csv_path}")
summary_df

Summary written to /Users/annavisman/stack/TUDelft/thesis/msc-thesis/src/thesis/baselines/results/ait_ads_fox_fixed_window_baseline_summary.csv


,method,condition,auc,precision,recall,f1
0,"Anomaly (IsolationForest, benign-only, fixed_w...",(no training),0.820226,0.542348,0.695575,0.609398
1,"Anomaly (OneClassSVM, benign-only, fixed_window)",(no training),0.895237,0.470414,0.703540,0.563830
2,"Anomaly + Mining (IsolationForest, benign-only...",(no training),0.811036,0.500972,0.597345,0.544240
3,"Anomaly + Mining (OneClassSVM, benign-only, fi...",(no training),0.793950,0.511401,0.694690,0.589118
4,"BERT (tokens as text, fixed_window)",Class-weighted (natural-ratio),NaN,1.000000,0.657522,0.793374
5,"Base schema (LogReg, fixed_window)",Class-weighted (natural-ratio),NaN,0.857955,0.668142,0.751244
6,"Base schema (RF, fixed_window)",Class-weighted (natural-ratio),NaN,0.423795,0.693805,0.526177
7,"Base schema (XGBoost, fixed_window)",Class-weighted (natural-ratio),NaN,0.384804,0.694690,0.495268
8,"Base schema + Mining (LogReg, fixed_window)",Class-weighted (natural-ratio),NaN,0.763819,0.672566,0.715294
9,"Base schema + Mining (RF, fixed_window)",Class-weighted (natural-ratio),NaN,0.429363,0.685841,0.528109


## LaTeX summary table

One block per baseline, with the configuration knobs that vary across baselines
broken out as columns:

| knob | column | values |
|---|---|---|
| input representation | **Features** | `Base (5)` reduced tabular schema · `Base (5) + mined` reduced schema plus attribute-mined symbolic features · `tokens as text` the `sig:`/`host:`/`short:` tokens serialized to text/prompt |
| training regime | **Training** | one row per **sampling condition** (`random` undersampling · `class-weighted` natural-ratio) for the trainable classifiers; `benign-only` for the one-class anomaly detectors; `none` for the zero-shot LLMs |
| learner | **Model** | RF / LogReg / XGBoost / DistilBERT / SecureBERT 2.0 / the four zero-shot models (`ZEROSHOT_MODEL_SLUGS`) / OneClassSVM / IsolationForest |

**Precision / Recall / F1** are at each model's own decision threshold, shown as
mean$_{\pm\text{sd}}$ over the 5 seeds. The four zero-shot rows and the OneClassSVM
anomaly rows are single deterministic runs (OneClassSVM shown as $\pm$0.000); the
IsolationForest anomaly rows are 5-seeded like the classifiers. All rows are
scored on this scenario's full test split. No `guided` condition and no
paper-target row -- both are CSCAS-only (see the intro cell).

The zero-shot block lists all four models in `ZEROSHOT_MODEL_SLUGS`; a model whose
`results/*.json` isn't present for this `(scenario, grouping)` renders as a blank
row (run `shell-scripts/baselines/run_ait_ads_zeroshot.sh`).

Writes `results/ait_ads_{SCENARIO}_{GROUPING_METHOD}_baseline_summary.tex` (the
`\resizebox` in the output needs `\usepackage{graphicx}`). `table_rows` from this
cell feeds the alert-volume table below.

In [6]:
# One block per baseline, with the config knobs that vary across baselines
# (input representation / training regime / learner) broken out as columns.
# Mirrors the CSCAS notebook's LaTeX-summary cell, minus the paper-target row
# (CSCAS-only), with two training conditions instead of three, and the
# zero-shot block expanded to the full ZEROSHOT_MODEL_SLUGS sweep.
# `table_rows` is reused by the alert-volume cell.
#
#   kind "trainable" -> one row per sampling condition (random / class_weighted)
#   kind "flat"      -> single row from the top-level metrics (zero-shot / anomaly)
ZEROSHOT_DISPLAY = {  # pretty names for the LaTeX Model column; slug used as-is otherwise
    "llama3.1-8b": "Llama-3.1-8B", "llama3.1-70b": "Llama-3.1-70B",
    "qwen2.5-7b": "Qwen2.5-7B", "qwen2.5-72b": "Qwen2.5-72B",
}

TABLE_SPEC = [  # (section, baseline, model, features, kind, result_key, regime_label)
    ("internal", "Internal system",   "Logistic Regression", "Base (5)",         "trainable", f"ait_ads_logreg_{GROUPING_METHOD}_{SCENARIO}",        None),
    ("internal", "Internal system",   "Random Forest",       "Base (5)",         "trainable", f"ait_ads_rf_{GROUPING_METHOD}_{SCENARIO}",            None),
    ("internal", "Internal system",   "XGBoost",             "Base (5)",         "trainable", f"ait_ads_xgboost_{GROUPING_METHOD}_{SCENARIO}",       None),
    ("mining",   "Internal + mining", "Random Forest",       "Base (5) + mined", "trainable", f"ait_ads_mining_{GROUPING_METHOD}_{SCENARIO}",        None),
    ("mining",   "Internal + mining", "Logistic Regression", "Base (5) + mined", "trainable", f"ait_ads_mining_logreg_{GROUPING_METHOD}_{SCENARIO}", None),
    ("mining",   "Internal + mining", "XGBoost",             "Base (5) + mined", "trainable", f"ait_ads_mining_xgboost_{GROUPING_METHOD}_{SCENARIO}", None),
    *[
        ("llm", "LLM (zero-shot)", ZEROSHOT_DISPLAY.get(s, s), "tokens as text", "flat", k, "none")
        for s, k in zip(ZEROSHOT_MODEL_SLUGS, _ZS)
    ],
    ("llm",      "LLM (general)",         "DistilBERT",          "tokens as text",   "trainable", f"ait_ads_bert_{GROUPING_METHOD}_{SCENARIO}",       None),
    ("llm",      "LLM (domain-specific)", "SecureBERT 2.0",      "tokens as text",   "trainable", f"ait_ads_securebert_{GROUPING_METHOD}_{SCENARIO}", None),
    ("anom",     "Anomaly",               "OneClassSVM",         "Base (5)",         "flat",      f"ait_ads_anomaly_{GROUPING_METHOD}_{SCENARIO}_ocsvm",         "benign-only"),
    ("anom",     "Anomaly",               "IsolationForest",     "Base (5)",         "flat",      f"ait_ads_anomaly_{GROUPING_METHOD}_{SCENARIO}_iforest",       "benign-only"),
    ("anom",     "Anomaly + mining",      "OneClassSVM",         "Base (5) + mined", "flat",      f"ait_ads_mining_anomaly_{GROUPING_METHOD}_{SCENARIO}_ocsvm",  "benign-only"),
    ("anom",     "Anomaly + mining",      "IsolationForest",     "Base (5) + mined", "flat",      f"ait_ads_mining_anomaly_{GROUPING_METHOD}_{SCENARIO}_iforest", "benign-only"),
]

TRAIN_CONDITIONS = [
    ("random", "random"),
    ("class_weighted", "class-weighted"),
]


def _pick(src: dict | None) -> dict | None:
    if src is None:
        return None
    out = {m: src.get(m) for m in ("precision", "recall", "f1")}
    seeds = src.get("seeds") or []
    # A flat row with no seed list is the OneClassSVM detector (deterministic
    # convex fit -- its run-to-run sd is genuinely 0.000) or a zero-shot row
    # (single run, no sd). is_anomaly() tells them apart.
    zero_sd = src.get("kind") == "anomaly"
    for m in ("precision", "recall", "f1"):
        vals = [s[m] for s in seeds]
        if len(vals) > 1:
            out[f"{m}_sd"] = float(np.std(vals, ddof=1))
        else:
            out[f"{m}_sd"] = 0.0 if zero_sd else None
    return out


table_rows = []
for gid, (section, baseline, model, features, kind, key, regime) in enumerate(TABLE_SPEC):
    data = results.get(key)
    if kind == "flat":
        entries = [(regime, _pick(data))]
    else:  # trainable -- one entry per sampling condition
        entries = [
            (label, _pick(data.get(cond) if data else None))
            for cond, label in TRAIN_CONDITIONS
        ]
    for training_label, m in entries:
        row = {"gid": gid, "section": section, "Baseline": baseline,
               "Model": model, "Features": features, "Training": training_label}
        for col in ("Precision", "Recall", "F1"):
            row[col] = None if m is None else m[col.lower()]
            row[f"{col}_sd"] = None if m is None else m.get(f"{col.lower()}_sd")
        table_rows.append(row)

table_df = pd.DataFrame(table_rows).drop(columns=["gid", "section"]).round(4)


# --- emit LaTeX (repeated Baseline/Model/Features blanked within a block) ---
def _cell(v, sd=None) -> str:
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return ""
    if sd is None or pd.isna(sd):
        return f"{v:.3f}"
    return f"${v:.3f}_{{\\pm {sd:.3f}}}$"


tex_lines = []
prev_section = prev_gid = None
for row in table_rows:
    if prev_section is not None and row["section"] != prev_section:
        tex_lines.append(r"\midrule")
    elif prev_gid is not None and row["gid"] != prev_gid:
        tex_lines.append(r"\addlinespace")
    first = row["gid"] != prev_gid
    prev_section, prev_gid = row["section"], row["gid"]
    b = row["Baseline"] if first else ""
    mo = row["Model"] if first else ""
    fe = row["Features"] if first else ""
    tex_lines.append(
        f"{b} & {mo} & {fe} & {row['Training']} & "
        f"{_cell(row['Precision'], row['Precision_sd'])} & "
        f"{_cell(row['Recall'], row['Recall_sd'])} & "
        f"{_cell(row['F1'], row['F1_sd'])}" + r" \\"
    )

_gm_tex = GROUPING_METHOD.replace("_", r"\_")
latex = (
    r"""\begin{table}[htbp]
\centering
\small
\caption{Summary of baseline performance on the AIT-ADS """ + f"{SCENARIO}" + r""" scenario,
grouping \texttt{""" + _gm_tex + r"""}. Trainable classifiers report both training-pool
sampling conditions (random undersampling, class-weighted natural-ratio) as
mean$_{\pm\text{sd}}$ over 5 seeds; the anomaly detectors are fit on benign rows
only and the zero-shot LLMs have no training step. Precision/recall/F1 are at each
model's own decision threshold. All rows are scored on this scenario's full test
split. The anomaly rows are at the detectors' default rejection rate ($\nu$ /
contamination $= 0.05$); Table~\ref{tab:ait-ads-""" + f"{SCENARIO}-{GROUPING_METHOD}" + r"""-anomaly-operating-points}
reports them at a tuned threshold. No \emph{guided} condition and no paper-target
row -- both are CSCAS-only.}
\label{tab:ait-ads-""" + f"{SCENARIO}-{GROUPING_METHOD}" + r"""-baseline-summary}
\resizebox{\textwidth}{!}{%
\begin{tabular}{llllccc}
\toprule
\textbf{Baseline} & \textbf{Model} & \textbf{Features} & \textbf{Training} & \textbf{Precision} & \textbf{Recall} & \textbf{F1} \\
\midrule
"""
    + "\n".join(tex_lines)
    + r"""
\bottomrule
\end{tabular}%
}
\end{table}
"""
)

latex_path = RESULTS_DIR / f"ait_ads_{SCENARIO}_{GROUPING_METHOD}_baseline_summary.tex"
latex_path.write_text(latex, encoding="utf-8")
print(f"LaTeX table written to {latex_path}\n")
print(latex)
table_df

LaTeX table written to /Users/annavisman/stack/TUDelft/thesis/msc-thesis/src/thesis/baselines/results/ait_ads_fox_fixed_window_baseline_summary.tex

\begin{table}[htbp]
\centering
\small
\caption{Summary of baseline performance on the AIT-ADS fox scenario,
grouping \texttt{fixed\_window}. Trainable classifiers report both training-pool
sampling conditions (random undersampling, class-weighted natural-ratio) as
mean$_{\pm\text{sd}}$ over 5 seeds; the anomaly detectors are fit on benign rows
only and the zero-shot LLMs have no training step. Precision/recall/F1 are at each
model's own decision threshold. All rows are scored on this scenario's full test
split. The anomaly rows are at the detectors' default rejection rate ($\nu$ /
contamination $= 0.05$); Table~\ref{tab:ait-ads-fox-fixed_window-anomaly-operating-points}
reports them at a tuned threshold. No \emph{guided} condition and no paper-target
row -- both are CSCAS-only.}
\label{tab:ait-ads-fox-fixed_window-baseline-summary}
\resize

,Baseline,Model,Features,Training,Precision,Precision_sd,Recall,Recall_sd,F1,F1_sd
0,Internal system,Logistic Regression,Base (5),random,0.8073,0.0940,0.6646,0.0037,0.7270,0.0350
1,Internal system,Logistic Regression,Base (5),class-weighted,0.8580,0.0000,0.6681,0.0000,0.7512,0.0000
2,Internal system,Random Forest,Base (5),random,0.4037,0.0049,0.7398,0.0145,0.5223,0.0059
3,Internal system,Random Forest,Base (5),class-weighted,0.4238,0.0025,0.6938,0.0037,0.5262,0.0023
4,Internal system,XGBoost,Base (5),random,0.4077,0.0082,0.7451,0.0202,0.5270,0.0113
5,Internal system,XGBoost,Base (5),class-weighted,0.3848,0.0000,0.6947,0.0000,0.4953,0.0000
6,Internal + mining,Random Forest,Base (5) + mined,random,0.4025,0.0055,0.7336,0.0096,0.5198,0.0061
7,Internal + mining,Random Forest,Base (5) + mined,class-weighted,0.4294,0.0000,0.6858,0.0000,0.5281,0.0000
8,Internal + mining,Logistic Regression,Base (5) + mined,random,0.7957,0.0970,0.6673,0.0048,0.7236,0.0344
9,Internal + mining,Logistic Regression,Base (5) + mined,class-weighted,0.7638,0.0000,0.6726,0.0000,0.7153,0.0000


## Alert volume / confusion counts

`precision` and `recall` become a concrete analyst load once the class balance is
fixed. This table covers **every row of the summary table above** (all sampling
conditions, all models). TP/FP/FN are on this scenario's full test split;
**Est. FP/day** projects the benign false-positive rate onto the test window's
wall-clock span.

The test-split composition and span are reconstructed **from the cached
`alert_groups`** (`CACHE_DIR/<scenario>/groups/<grouping_method>/alert_groups/alert_groups_raw.json`)
-- the same last-`TEST_FRAC` chronological slice `load_ait_ads_baseline_split`
takes, no pipeline re-run. `alertbert`/`deepcase` groupings whose cache isn't
present locally fall back to an anomaly result's confusion counts for the
composition, and `Est. FP/day` shows `--` (the span needs the grouped data). The
`always-alert` row is the ceiling. Writes
`results/ait_ads_{SCENARIO}_{GROUPING_METHOD}_alert_volume.tex`.

In [7]:
# Confusion counts + alert volume for every row of the summary table above,
# derived from its (precision, recall) and the test-split composition.
#
# The test-split composition and its wall-clock span are reconstructed from the
# cached alert_groups -- no pipeline re-run: read the cached AlertGroup list,
# keep the benign/attack-labelled ones, take the last TEST_FRAC in cache order
# (the cache is written start_ts-sorted). This is exactly the slice
# _ait_ads_data.load_ait_ads_baseline_split takes. alertbert/deepcase groupings
# whose cache isn't present locally fall back to an anomaly result's own
# tp/fp/tn/fn (its workload_at_recall) for the composition, and "Est. FP/day"
# is left blank (the span needs the grouped data).
TEST_FRAC = 0.3  # matches load_ait_ads_baseline_split's default


def _split_from_cache():
    raw = (
        CACHE_DIR / SCENARIO / "groups" / GROUPING_METHOD
        / "alert_groups" / "alert_groups_raw.json"
    )
    if not raw.exists():
        return None
    groups = json.loads(raw.read_text())
    labelled = [g for g in groups if g.get("group_label") in ("benign", "attack")]
    if not labelled:
        return None
    test = labelled[int(len(labelled) * (1 - TEST_FRAC)):]
    pos = sum(g["group_label"] == "attack" for g in test)
    neg = sum(g["group_label"] == "benign" for g in test)
    starts = [g["start_ts"] for g in test]
    ends = [g["end_ts"] if g.get("end_ts") is not None else g["start_ts"] for g in test]
    days = (max(ends) - min(starts)) / 86_400.0
    return pos, neg, (days if days > 0 else None), "cached alert_groups"


def _split_from_anomaly():
    for name, data in results.items():
        if not is_anomaly(data):
            continue
        for tgt in ("0.90", "0.95", "0.99"):
            wl = (data.get("workload_at_recall") or {}).get(tgt)
            if wl:
                return round(wl["tp"] + wl["fn"]), round(wl["fp"] + wl["tn"]), None, f"{name} workload_at_recall"
    return None


_info = _split_from_cache() or _split_from_anomaly()
_vol_cols = ["Baseline", "Model", "Training", "TP", "FP", "FN", "Precision", "Recall", "Est. FP/day"]
if _info is None or not table_rows:
    print("No test-split composition available -- need the cached alert_groups or a loaded anomaly result.")
    vol_df = pd.DataFrame(columns=_vol_cols)
else:
    TEST_POS, TEST_NEG, TEST_DAYS, _src = _info
    TEST_PREVALENCE = TEST_POS / (TEST_POS + TEST_NEG)
    _span = f"{TEST_DAYS:.2f} days" if TEST_DAYS else "timespan unavailable (grouping cache absent)"
    print(f"Test split: {TEST_POS} attack / {TEST_NEG} benign  |  {_span}  [{_src}]\n")

    def _counts(p, r):
        tp = round(r * TEST_POS)
        fn = TEST_POS - tp
        fp = TEST_NEG if not p else round(tp * (1 - p) / p)
        return tp, fp, fn

    def _fp_day(fp):
        return None if not TEST_DAYS else fp / TEST_DAYS

    vol_rows, vol_tex = [], []
    prev_section = prev_gid = None
    for row in table_rows:  # built by the LaTeX-summary cell above
        p, r = row["Precision"], row["Recall"]
        if p is None or r is None or (isinstance(r, float) and pd.isna(r)):
            continue
        if prev_section is not None and row["section"] != prev_section:
            vol_tex.append(r"\midrule")
        elif prev_gid is not None and row["gid"] != prev_gid:
            vol_tex.append(r"\addlinespace")
        first = row["gid"] != prev_gid
        prev_section, prev_gid = row["section"], row["gid"]

        tp, fp, fn = _counts(p, r)
        fpd = _fp_day(fp)
        vol_rows.append({"Baseline": row["Baseline"], "Model": row["Model"],
                         "Training": row["Training"], "TP": tp, "FP": fp, "FN": fn,
                         "Precision": round(p, 3), "Recall": round(r, 3),
                         "Est. FP/day": None if fpd is None else round(fpd, 1)})
        b = row["Baseline"] if first else ""
        mo = row["Model"] if first else ""
        vol_tex.append(
            f"{b} & {mo} & {row['Training']} & {tp} & {fp} & {fn} & "
            f"{p:.3f} & {r:.3f} & " + ("--" if fpd is None else f"{fpd:,.1f}") + r" \\"
        )

    # always-alert ceiling
    _tp, _fp, _fn = _counts(TEST_PREVALENCE, 1.0)
    _fpd = _fp_day(_fp)
    vol_rows.append({"Baseline": "always-alert", "Model": "--", "Training": "--",
                     "TP": _tp, "FP": _fp, "FN": _fn, "Precision": round(TEST_PREVALENCE, 3),
                     "Recall": 1.0, "Est. FP/day": None if _fpd is None else round(_fpd, 1)})
    vol_tex.append(r"\midrule")
    vol_tex.append(
        f"always-alert & -- & -- & {_tp} & {_fp} & {_fn} & "
        f"{TEST_PREVALENCE:.3f} & 1.000 & " + ("--" if _fpd is None else f"{_fpd:,.1f}") + r" \\"
    )

    vol_df = pd.DataFrame(vol_rows)

    _gm_tex = GROUPING_METHOD.replace("_", r"\_")
    _days_txt = f"{TEST_DAYS:.2f}" if TEST_DAYS else "n/a"
    _neg_txt = f"{TEST_NEG:,}".replace(",", "{,}")
    vol_latex = (
        r"""\begin{table}[htbp]
\centering
\small
\caption{Confusion-count and alert-volume view of every row in
Table~\ref{tab:ait-ads-""" + f"{SCENARIO}-{GROUPING_METHOD}" + r"""-baseline-summary} (AIT-ADS
"""
        + f"{SCENARIO}" + r""" scenario, grouping \texttt{""" + _gm_tex + r"""}). TP/FP/FN are on
this scenario's full test split (""" + f"{TEST_POS}" + r""" attack / """ + _neg_txt + r""" benign
alert\_groups) -- precision and recall projected onto that fixed composition.
\emph{Est.\ FP/day} scales the benign false-positive rate to the test window's
wall-clock span (${\sim}$""" + _days_txt + r""" days) -- the analyst-load axis F1 hides.
The \texttt{always-alert} row is the ceiling.}
\label{tab:ait-ads-""" + f"{SCENARIO}-{GROUPING_METHOD}" + r"""-alert-volume}
\resizebox{\textwidth}{!}{%
\begin{tabular}{lllrrrccr}
\toprule
\textbf{Baseline} & \textbf{Model} & \textbf{Training} & \textbf{TP} & \textbf{FP} & \textbf{FN} & \textbf{Precision} & \textbf{Recall} & \textbf{Est. FP/day} \\
\midrule
"""
        + "\n".join(vol_tex)
        + r"""
\bottomrule
\end{tabular}%
}
\end{table}
"""
    )

    vol_tex_path = RESULTS_DIR / f"ait_ads_{SCENARIO}_{GROUPING_METHOD}_alert_volume.tex"
    vol_tex_path.write_text(vol_latex, encoding="utf-8")
    print(f"Written to {vol_tex_path}\n")
    print(vol_latex)
vol_df

Test split: 226 attack / 2868 benign  |  1.47 days  [cached alert_groups]

Written to /Users/annavisman/stack/TUDelft/thesis/msc-thesis/src/thesis/baselines/results/ait_ads_fox_fixed_window_alert_volume.tex

\begin{table}[htbp]
\centering
\small
\caption{Confusion-count and alert-volume view of every row in
Table~\ref{tab:ait-ads-fox-fixed_window-baseline-summary} (AIT-ADS
fox scenario, grouping \texttt{fixed\_window}). TP/FP/FN are on
this scenario's full test split (226 attack / 2{,}868 benign
alert\_groups) -- precision and recall projected onto that fixed composition.
\emph{Est.\ FP/day} scales the benign false-positive rate to the test window's
wall-clock span (${\sim}$1.47 days) -- the analyst-load axis F1 hides.
The \texttt{always-alert} row is the ceiling.}
\label{tab:ait-ads-fox-fixed_window-alert-volume}
\resizebox{\textwidth}{!}{%
\begin{tabular}{lllrrrccr}
\toprule
\textbf{Baseline} & \textbf{Model} & \textbf{Training} & \textbf{TP} & \textbf{FP} & \textbf{FN} & \textbf{Pre

,Baseline,Model,Training,TP,FP,FN,Precision,Recall,Est. FP/day
0,Internal system,Logistic Regression,random,150,36,76,0.807,0.665,24.4
1,Internal system,Logistic Regression,class-weighted,151,25,75,0.858,0.668,17.0
2,Internal system,Random Forest,random,167,247,59,0.404,0.740,167.6
3,Internal system,Random Forest,class-weighted,157,213,69,0.424,0.694,144.5
4,Internal system,XGBoost,random,168,244,58,0.408,0.745,165.5
5,Internal system,XGBoost,class-weighted,157,251,69,0.385,0.695,170.3
6,Internal + mining,Random Forest,random,166,246,60,0.402,0.734,166.9
7,Internal + mining,Random Forest,class-weighted,155,206,71,0.429,0.686,139.7
8,Internal + mining,Logistic Regression,random,151,39,75,0.796,0.667,26.5
9,Internal + mining,Logistic Regression,class-weighted,152,47,74,0.764,0.673,31.9


## Anomaly detectors at a tuned operating point

The plots above are at each detector's default rejection rate (`nu` /
`contamination` = 0.05). This table reports each at the threshold that hits
**90% recall** (`training.workload.compute_workload_at_recall`, seed-averaged for
the IsolationForest rows): the precision an analyst sees at that cut, the
false-positive count on this scenario's test split, and `Workload -` = the
fraction of alerts the model lets them skip while still catching 90% of attacks.
`AUC` is the threshold-free ceiling. Writes
`results/ait_ads_{SCENARIO}_{GROUPING_METHOD}_anomaly_operating_points.tex`.

In [8]:
ANOMALY_OP_SPEC = [  # (label, model, result_key_suffix, mining?)
    ("Anomaly",          "OneClassSVM",     f"ait_ads_anomaly_{GROUPING_METHOD}_{SCENARIO}_ocsvm"),
    ("Anomaly",          "IsolationForest", f"ait_ads_anomaly_{GROUPING_METHOD}_{SCENARIO}_iforest"),
    ("Anomaly + mining", "OneClassSVM",     f"ait_ads_mining_anomaly_{GROUPING_METHOD}_{SCENARIO}_ocsvm"),
    ("Anomaly + mining", "IsolationForest", f"ait_ads_mining_anomaly_{GROUPING_METHOD}_{SCENARIO}_iforest"),
]
OP_TARGET = "0.90"  # target recall; JSON also carries 0.95 / 0.99

op_rows, op_tex = [], []
for label, model, key in ANOMALY_OP_SPEC:
    data = results.get(key)
    if data is None:
        print(f"[skip] {key} -- run the script first")
        continue
    wl = (data.get("workload_at_recall") or {}).get(OP_TARGET)
    op_rows.append({
        "Baseline": label, "Model": model,
        "AUC": round(data["auc"], 3),
        "F1 (default cut)": round(data["f1"], 3),
        "P@R>=.90": None if wl is None else round(wl["precision"], 3),
        "FP@R>=.90": None if wl is None else int(round(wl["fp"])),
        "Workload-@R>=.90": None if wl is None else round(wl["workload_reduction"], 3),
    })
    if wl is None:
        op_tex.append(f"{label} & {model} & {data['auc']:.3f} & {data['f1']:.3f} & -- & -- & --" + r" \\")
    else:
        op_tex.append(
            f"{label} & {model} & {data['auc']:.3f} & {data['f1']:.3f} & "
            f"{wl['precision']:.3f} & {wl['fp']:,.0f} & {wl['workload_reduction']:.3f}" + r" \\"
        )

op_df = pd.DataFrame(op_rows)

op_latex = (
    r"""\begin{table}[htbp]
\centering
\small
\caption{AIT-ADS anomaly detectors at a tuned operating point (scenario """
    + f"{SCENARIO}" + r""", grouping """ + f"{GROUPING_METHOD}" + r"""). \textbf{AUC} is
threshold-free ranking quality; \textbf{F1 (default cut)} is at the detector's own
$\nu$ / contamination $= 0.05$ rejection rate. The last three columns report the
detector at the threshold achieving 90\% recall: the precision at that cut, the
false-positive count on this scenario's test split, and \emph{Workload $-$} = the
fraction of alerts the analyst can skip while still catching 90\% of attacks.
IsolationForest rows are seed-averaged (5 seeds).}
\label{tab:ait-ads-""" + f"{SCENARIO}-{GROUPING_METHOD}" + r"""-anomaly-operating-points}
\begin{tabular}{llccrrr}
\toprule
\textbf{Baseline} & \textbf{Model} & \textbf{AUC} & \textbf{F1 (def.\ cut)} & \textbf{P@R$\geq$.90} & \textbf{FP@R$\geq$.90} & \textbf{Workload$-$@R$\geq$.90} \\
\midrule
"""
    + "\n".join(op_tex)
    + r"""
\bottomrule
\end{tabular}
\end{table}
"""
)

op_tex_path = RESULTS_DIR / f"ait_ads_{SCENARIO}_{GROUPING_METHOD}_anomaly_operating_points.tex"
op_tex_path.write_text(op_latex, encoding="utf-8")
print(f"Written to {op_tex_path}\n")
print(op_latex)
op_df

Written to /Users/annavisman/stack/TUDelft/thesis/msc-thesis/src/thesis/baselines/results/ait_ads_fox_fixed_window_anomaly_operating_points.tex

\begin{table}[htbp]
\centering
\small
\caption{AIT-ADS anomaly detectors at a tuned operating point (scenario fox, grouping fixed_window). \textbf{AUC} is
threshold-free ranking quality; \textbf{F1 (default cut)} is at the detector's own
$\nu$ / contamination $= 0.05$ rejection rate. The last three columns report the
detector at the threshold achieving 90\% recall: the precision at that cut, the
false-positive count on this scenario's test split, and \emph{Workload $-$} = the
fraction of alerts the analyst can skip while still catching 90\% of attacks.
IsolationForest rows are seed-averaged (5 seeds).}
\label{tab:ait-ads-fox-fixed_window-anomaly-operating-points}
\begin{tabular}{llccrrr}
\toprule
\textbf{Baseline} & \textbf{Model} & \textbf{AUC} & \textbf{F1 (def.\ cut)} & \textbf{P@R$\geq$.90} & \textbf{FP@R$\geq$.90} & \textbf{Workload$-$@R$

,Baseline,Model,AUC,F1 (default cut),P@R>=.90,FP@R>=.90,Workload-@R>=.90
0,Anomaly,OneClassSVM,0.895,0.564,0.205,812,0.670
1,Anomaly,IsolationForest,0.820,0.609,0.074,2810,0.019
2,Anomaly + mining,OneClassSVM,0.794,0.589,0.075,2759,0.036
3,Anomaly + mining,IsolationForest,0.811,0.544,0.076,2745,0.040
